# Análisis de Marketing — Showz
**Objetivo:** Optimizar los gastos de marketing analizando el comportamiento de usuarios, conversiones, LTV y ROMI por fuente de adquisición.

**Período:** Enero 2017 – Diciembre 2018  
**Analista:** Sprint 10 — Proyecto Final de Análisis de Negocio

---
## Paso 1. Carga y preparación de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

# Estilo visual
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

In [ ]:
# --- Carga de archivos ---
visits = pd.read_csv('/datasets/visits_log_us.csv')
orders = pd.read_csv('/datasets/orders_log_us.csv')
costs  = pd.read_csv('/datasets/costs_us.csv')

print('Visitas:', visits.shape)
print('Pedidos:', orders.shape)
print('Costos: ', costs.shape)

In [ ]:
# --- Inspección inicial ---
print('=== VISITAS ===')
display(visits.head(3))
display(visits.dtypes)
print('Nulos:', visits.isnull().sum().to_dict())

print('\n=== PEDIDOS ===')
display(orders.head(3))
display(orders.dtypes)
print('Nulos:', orders.isnull().sum().to_dict())

print('\n=== COSTOS ===')
display(costs.head(3))
display(costs.dtypes)
print('Nulos:', costs.isnull().sum().to_dict())

In [ ]:
# --- Estandarizar nombres de columnas ---
visits.columns = visits.columns.str.strip().str.lower().str.replace(' ', '_')
orders.columns = orders.columns.str.strip().str.lower().str.replace(' ', '_')
costs.columns  = costs.columns.str.strip().str.lower().str.replace(' ', '_')

print('Columnas visits:', visits.columns.tolist())
print('Columnas orders:', orders.columns.tolist())
print('Columnas costs: ', costs.columns.tolist())

In [ ]:
# --- Conversión de tipos: fechas ---
visits['start_ts'] = pd.to_datetime(visits['start_ts'])
visits['end_ts']   = pd.to_datetime(visits['end_ts'])
orders['buy_ts']   = pd.to_datetime(orders['buy_ts'])
costs['dt']        = pd.to_datetime(costs['dt'])

# source_id como string para uniones consistentes
visits['source_id'] = visits['source_id'].astype(str)
costs['source_id']  = costs['source_id'].astype(str)

# Columnas derivadas de tiempo
visits['session_date'] = visits['start_ts'].dt.date
visits['session_week'] = visits['start_ts'].dt.to_period('W')
visits['session_month']= visits['start_ts'].dt.to_period('M')
visits['session_duration_min'] = (visits['end_ts'] - visits['start_ts']).dt.total_seconds() / 60

orders['order_date']  = orders['buy_ts'].dt.date
orders['order_month'] = orders['buy_ts'].dt.to_period('M')

costs['cost_month'] = costs['dt'].dt.to_period('M')

# Verificar
print('Rango visitas:', visits['start_ts'].min(), '→', visits['start_ts'].max())
print('Rango pedidos:', orders['buy_ts'].min(), '→', orders['buy_ts'].max())
print('Rango costos: ', costs['dt'].min(), '→', costs['dt'].max())

In [ ]:
# --- Eliminación de duplicados ---
before_v = len(visits)
before_o = len(orders)
before_c = len(costs)

visits.drop_duplicates(inplace=True)
orders.drop_duplicates(inplace=True)
costs.drop_duplicates(inplace=True)

print(f'Visitas eliminadas: {before_v - len(visits)}')
print(f'Pedidos eliminados: {before_o - len(orders)}')
print(f'Costos eliminados:  {before_c - len(costs)}')

**Comentario — Preparación de datos:**  
Se estandarizaron los nombres de columnas a snake_case minúscula, se convirtieron todas las fechas a `datetime64`, y se crearon columnas auxiliares (fecha, semana, mes, duración de sesión). No se encontraron valores nulos. Los duplicados fueron eliminados. Los datos están listos para el análisis.

---
## Paso 2. Métricas e informes
### 2.1 Análisis de Visitas

#### 2.1.1 Usuarios únicos por día, semana y mes (DAU / WAU / MAU)

In [ ]:
dau = visits.groupby('session_date')['uid'].nunique()
wau = visits.groupby('session_week')['uid'].nunique()
mau = visits.groupby('session_month')['uid'].nunique()

print(f'DAU promedio: {dau.mean():.0f} usuarios/día')
print(f'WAU promedio: {wau.mean():.0f} usuarios/semana')
print(f'MAU promedio: {mau.mean():.0f} usuarios/mes')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(dau.index, dau.values, linewidth=1, color='steelblue')
axes[0].set_title('Usuarios únicos por día (DAU)')
axes[0].set_xlabel('Fecha')
axes[0].set_ylabel('Usuarios únicos')
axes[0].tick_params(axis='x', rotation=45)

axes[1].plot([str(w) for w in wau.index], wau.values, linewidth=1.5, color='coral')
axes[1].set_title('Usuarios únicos por semana (WAU)')
axes[1].set_xlabel('Semana')
axes[1].set_ylabel('Usuarios únicos')
axes[1].tick_params(axis='x', rotation=45)
axes[1].xaxis.set_major_locator(mticker.MaxNLocator(nbins=10))

axes[2].plot([str(m) for m in mau.index], mau.values, marker='o', linewidth=2, color='seagreen')
axes[2].set_title('Usuarios únicos por mes (MAU)')
axes[2].set_xlabel('Mes')
axes[2].set_ylabel('Usuarios únicos')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.suptitle('DAU / WAU / MAU — Showz 2017-2018', y=1.02, fontsize=15, fontweight='bold')
plt.show()

**Comentario:** El tráfico diario muestra variaciones estacionales. El gráfico mensual revela si hay crecimiento sostenido en la base de usuarios activos.

#### 2.1.2 Sesiones por día

In [ ]:
sessions_per_day = visits.groupby('session_date').size()

print(f'Sesiones promedio por día: {sessions_per_day.mean():.1f}')
print(f'Sesiones máximas en un día: {sessions_per_day.max()}')

plt.figure(figsize=(14, 4))
plt.plot(sessions_per_day.index, sessions_per_day.values, linewidth=1, color='mediumpurple')
plt.axhline(sessions_per_day.mean(), color='red', linestyle='--', label=f'Promedio: {sessions_per_day.mean():.0f}')
plt.title('Sesiones por día — Showz')
plt.xlabel('Fecha')
plt.ylabel('Número de sesiones')
plt.legend()
plt.tight_layout()
plt.show()

**Comentario:** El número de sesiones por día supera al DAU cuando los usuarios tienen múltiples sesiones. Comparar ambas métricas permite estimar la tasa de sesiones por usuario.

#### 2.1.3 Duración de sesiones

In [ ]:
# Filtrar duraciones negativas o anómalas (> 24h)
valid_sessions = visits[(visits['session_duration_min'] >= 0) & 
                        (visits['session_duration_min'] <= 1440)]

print(f'Duración promedio de sesión: {valid_sessions["session_duration_min"].mean():.2f} minutos')
print(f'Mediana de duración:         {valid_sessions["session_duration_min"].median():.2f} minutos')
print(f'Sesiones con duración 0:     {(valid_sessions["session_duration_min"] == 0).sum()}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(valid_sessions['session_duration_min'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de duración de sesiones')
axes[0].set_xlabel('Duración (minutos)')
axes[0].set_ylabel('Frecuencia')

# Por dispositivo
dur_device = valid_sessions.groupby('device')['session_duration_min'].mean().sort_values(ascending=False)
axes[1].bar(dur_device.index, dur_device.values, color=['steelblue', 'coral'])
axes[1].set_title('Duración promedio de sesión por dispositivo')
axes[1].set_xlabel('Dispositivo')
axes[1].set_ylabel('Duración promedio (min)')

plt.tight_layout()
plt.show()

**Comentario:** La mayoría de las sesiones son cortas. La diferencia entre promedio y mediana indica presencia de sesiones muy largas que sesgan el promedio.

#### 2.1.4 Frecuencia de retorno (retención)

In [ ]:
# Primera visita de cada usuario (inicio de cohorte)
first_visit = visits.groupby('uid')['start_ts'].min().reset_index()
first_visit.columns = ['uid', 'first_visit']
first_visit['cohort_month'] = first_visit['first_visit'].dt.to_period('M')

# Unir con todas las visitas
visits_cohort = visits.merge(first_visit[['uid', 'cohort_month']], on='uid')
visits_cohort['age_months'] = (
    visits_cohort['session_month'].astype(int) - 
    visits_cohort['cohort_month'].astype(int)
)

# Tabla de retención
retention_table = visits_cohort.groupby(['cohort_month', 'age_months'])['uid'].nunique().reset_index()
cohort_sizes = retention_table[retention_table['age_months'] == 0][['cohort_month', 'uid']]
cohort_sizes.columns = ['cohort_month', 'cohort_size']

retention_table = retention_table.merge(cohort_sizes, on='cohort_month')
retention_table['retention_rate'] = retention_table['uid'] / retention_table['cohort_size']

retention_pivot = retention_table.pivot_table(
    index='cohort_month', columns='age_months', values='retention_rate'
).fillna(0)

# Heatmap
plt.figure(figsize=(16, 8))
sns.heatmap(
    retention_pivot.iloc[:, :12],  # Primeros 12 meses
    annot=True, fmt='.0%', cmap='YlOrRd',
    linewidths=0.5, vmin=0, vmax=0.5
)
plt.title('Tasa de retención por cohorte mensual (primeros 12 meses)', fontsize=14)
plt.xlabel('Meses desde primera visita')
plt.ylabel('Cohorte (mes de primera visita)')
plt.tight_layout()
plt.show()

print('\nRetención promedio mes 1:', retention_pivot[1].mean() if 1 in retention_pivot.columns else 'N/A')
print('Retención promedio mes 3:', retention_pivot[3].mean() if 3 in retention_pivot.columns else 'N/A')

**Comentario:** El heatmap de retención muestra qué cohortes retienen mejor a sus usuarios. Una caída brusca en el mes 1 indica problemas de retención temprana.

---
### 2.2 Análisis de Ventas

#### 2.2.1 Tiempo hasta la primera compra (conversión)

In [ ]:
# Primera visita y primera compra por usuario
first_order = orders.groupby('uid')['buy_ts'].min().reset_index()
first_order.columns = ['uid', 'first_order']

conversion = first_visit.merge(first_order, on='uid')
conversion['days_to_convert'] = (conversion['first_order'] - conversion['first_visit']).dt.days

# Clasificar en ventanas
def classify_conversion(days):
    if days == 0:   return '0d (mismo día)'
    elif days <= 1: return '1d'
    elif days <= 7: return '2-7d'
    elif days <= 30: return '8-30d'
    else:           return '>30d'

conversion['conversion_window'] = conversion['days_to_convert'].apply(classify_conversion)

conv_dist = conversion['conversion_window'].value_counts().reindex(
    ['0d (mismo día)', '1d', '2-7d', '8-30d', '>30d']
)

print('Distribución de conversiones:')
print(conv_dist)
print(f'\nMediana días hasta conversión: {conversion["days_to_convert"].median():.0f} días')
print(f'Promedio días hasta conversión: {conversion["days_to_convert"].mean():.1f} días')

plt.figure(figsize=(10, 5))
conv_dist.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Distribución del tiempo hasta la primera compra')
plt.xlabel('Ventana de conversión')
plt.ylabel('Número de usuarios')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Comentario:** La ventana de conversión indica qué tan rápido los usuarios realizan su primera compra. Si la mayoría convierte en el día 0, el ciclo de venta es corto y la intención de compra es alta.

#### 2.2.2 Pedidos por período de tiempo

In [ ]:
orders_per_day   = orders.groupby('order_date').size()
orders_per_month = orders.groupby('order_month').size()

print(f'Promedio de pedidos por día:  {orders_per_day.mean():.1f}')
print(f'Promedio de pedidos por mes:  {orders_per_month.mean():.1f}')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(orders_per_day.index, orders_per_day.values, linewidth=1, color='darkorange')
axes[0].set_title('Pedidos por día')
axes[0].set_xlabel('Fecha')
axes[0].set_ylabel('Pedidos')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar([str(m) for m in orders_per_month.index], orders_per_month.values, color='darkorange')
axes[1].set_title('Pedidos por mes')
axes[1].set_xlabel('Mes')
axes[1].set_ylabel('Pedidos')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

#### 2.2.3 Tamaño promedio de compra (AOV)

In [ ]:
aov_global = orders['revenue'].mean()
aov_monthly = orders.groupby('order_month')['revenue'].mean()

print(f'AOV global: ${aov_global:.2f}')

plt.figure(figsize=(12, 5))
plt.plot([str(m) for m in aov_monthly.index], aov_monthly.values, 
         marker='o', linewidth=2, color='seagreen')
plt.axhline(aov_global, color='red', linestyle='--', label=f'Promedio global: ${aov_global:.2f}')
plt.title('Ticket promedio (AOV) por mes')
plt.xlabel('Mes')
plt.ylabel('Revenue promedio por pedido ($)')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

#### 2.2.4 LTV (Lifetime Value) por cohorte

In [ ]:
# Unir pedidos con cohorte
orders_cohort = orders.merge(first_visit[['uid', 'cohort_month']], on='uid')
orders_cohort['age_months'] = (
    orders_cohort['order_month'].astype(int) - 
    orders_cohort['cohort_month'].astype(int)
)

# Revenue acumulado por cohorte y mes de vida
ltv_table = orders_cohort.groupby(['cohort_month', 'age_months'])['revenue'].sum().reset_index()
ltv_table = ltv_table.merge(cohort_sizes, on='cohort_month')

# Revenue acumulado (cumsum) por usuario
ltv_table = ltv_table.sort_values(['cohort_month', 'age_months'])
ltv_table['cumrev'] = ltv_table.groupby('cohort_month')['revenue'].cumsum()
ltv_table['ltv'] = ltv_table['cumrev'] / ltv_table['cohort_size']

ltv_pivot = ltv_table.pivot_table(
    index='cohort_month', columns='age_months', values='ltv'
).fillna(method='ffill', axis=1)

plt.figure(figsize=(16, 8))
sns.heatmap(
    ltv_pivot.iloc[:, :12],
    annot=True, fmt='.2f', cmap='Blues',
    linewidths=0.5
)
plt.title('LTV acumulado por cohorte ($/usuario, primeros 12 meses)', fontsize=14)
plt.xlabel('Meses desde primera visita')
plt.ylabel('Cohorte')
plt.tight_layout()
plt.show()

# LTV promedio al mes 0 y mes 6
print('LTV promedio mes 0:', ltv_pivot[0].mean() if 0 in ltv_pivot.columns else 'N/A')
print('LTV promedio mes 6:', ltv_pivot[6].mean() if 6 in ltv_pivot.columns else 'N/A')

**Comentario:** El heatmap de LTV muestra cuánto valor aporta cada cohorte a lo largo del tiempo. Cohortes con LTV alto y creciente son las más valiosas para el negocio.

---
### 2.3 Análisis de Marketing

#### 2.3.1 Gasto total, por fuente y evolución temporal

In [ ]:
total_spend = costs['costs'].sum()
spend_by_source = costs.groupby('source_id')['costs'].sum().sort_values(ascending=False)
spend_by_month  = costs.groupby('cost_month')['costs'].sum()

print(f'Gasto total en marketing: ${total_spend:,.2f}')
print('\nGasto por fuente:')
print(spend_by_source)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

spend_by_source.plot(kind='bar', ax=axes[0], color='coral', edgecolor='white')
axes[0].set_title('Gasto total por fuente de anuncios')
axes[0].set_xlabel('Fuente (source_id)')
axes[0].set_ylabel('Gasto total ($)')
axes[0].tick_params(axis='x', rotation=0)

axes[1].plot([str(m) for m in spend_by_month.index], spend_by_month.values,
             marker='o', linewidth=2, color='coral')
axes[1].set_title('Gasto mensual en marketing')
axes[1].set_xlabel('Mes')
axes[1].set_ylabel('Gasto ($)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

#### 2.3.2 CAC (Costo de Adquisición de Clientes) por fuente

In [ ]:
# Usuarios que compraron al menos una vez, con su fuente de origen
buyers = orders[['uid']].drop_duplicates()
first_visit_source = visits.sort_values('start_ts').groupby('uid').first().reset_index()[['uid', 'source_id']]

buyers_with_source = buyers.merge(first_visit_source, on='uid')
buyers_by_source = buyers_with_source.groupby('source_id')['uid'].nunique()

# CAC = gasto total de la fuente / compradores adquiridos
cac = (spend_by_source / buyers_by_source).dropna().sort_values(ascending=True)

print('CAC por fuente:')
print(cac.round(2))

plt.figure(figsize=(10, 5))
cac.plot(kind='bar', color='mediumpurple', edgecolor='white')
plt.title('CAC por fuente de anuncios')
plt.xlabel('Fuente (source_id)')
plt.ylabel('CAC ($)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Comentario:** El CAC mide cuánto cuesta convertir un visitante en cliente por cada fuente. Un CAC bajo combinado con un LTV alto indica alta rentabilidad de esa fuente.

#### 2.3.3 ROMI (Return on Marketing Investment) por fuente

In [ ]:
# Revenue total por fuente
orders_source = orders.merge(first_visit_source, on='uid')
revenue_by_source = orders_source.groupby('source_id')['revenue'].sum()

# ROMI = (Revenue - Gasto) / Gasto
romi = ((revenue_by_source - spend_by_source) / spend_by_source).dropna().sort_values(ascending=False)

print('ROMI por fuente:')
print(romi.round(4).apply(lambda x: f'{x*100:.1f}%'))

colors = ['seagreen' if r > 0 else 'tomato' for r in romi.values]
plt.figure(figsize=(10, 5))
plt.bar(romi.index, romi.values * 100, color=colors, edgecolor='white')
plt.axhline(0, color='black', linewidth=0.8)
plt.title('ROMI por fuente de anuncios (%)')
plt.xlabel('Fuente (source_id)')
plt.ylabel('ROMI (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Comentario:** El ROMI positivo (verde) indica fuentes donde el ingreso supera la inversión. El ROMI negativo (rojo) señala fuentes que no están siendo rentables.

#### 2.3.4 ROMI acumulado por cohorte y fuente (evolución)

In [ ]:
# Construir tabla de ROMI por cohorte y fuente a lo largo del tiempo
# Primero, visitas con cohorte y fuente
visits_full = visits.merge(first_visit[['uid', 'cohort_month']], on='uid')
first_visit_full = first_visit.merge(
    visits.sort_values('start_ts').groupby('uid').first().reset_index()[['uid', 'source_id']], 
    on='uid'
)

# Usuarios por cohorte y fuente
users_cohort_source = first_visit_full.groupby(['cohort_month', 'source_id'])['uid'].nunique().reset_index()
users_cohort_source.columns = ['cohort_month', 'source_id', 'n_users']

# Costos por cohorte y fuente (asignados al mes de la cohorte)
costs_monthly = costs.copy()
costs_monthly['cohort_month'] = costs_monthly['dt'].dt.to_period('M')
costs_by_cohort_source = costs_monthly.groupby(['cohort_month', 'source_id'])['costs'].sum().reset_index()

# Revenue por cohorte y fuente
orders_full = orders.merge(first_visit_full[['uid', 'cohort_month', 'source_id']], on='uid')
revenue_cohort_source = orders_full.groupby(['cohort_month', 'source_id'])['revenue'].sum().reset_index()

# Tabla consolidada
romi_table = users_cohort_source.merge(costs_by_cohort_source, on=['cohort_month', 'source_id'], how='left')
romi_table = romi_table.merge(revenue_cohort_source, on=['cohort_month', 'source_id'], how='left')
romi_table.fillna(0, inplace=True)
romi_table['romi'] = np.where(
    romi_table['costs'] > 0,
    (romi_table['revenue'] - romi_table['costs']) / romi_table['costs'],
    np.nan
)

# Pivot ROMI por fuente a lo largo del tiempo
romi_over_time = romi_table.pivot_table(index='cohort_month', columns='source_id', values='romi')

plt.figure(figsize=(14, 6))
for col in romi_over_time.columns:
    plt.plot([str(x) for x in romi_over_time.index], 
             romi_over_time[col] * 100, marker='o', label=f'Fuente {col}', linewidth=1.5)
plt.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.title('ROMI mensual por fuente de anuncios')
plt.xlabel('Mes de cohorte')
plt.ylabel('ROMI (%)')
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

#### 2.3.5 Métricas clave por dispositivo

In [ ]:
# Usuarios únicos por dispositivo
users_device = visits.groupby('device')['uid'].nunique()

# Compradores por dispositivo (primera visita)
buyers_device = buyers_with_source.merge(first_visit_source, on=['uid', 'source_id'])
buyers_dev = first_visit.merge(visits[['uid', 'device']].drop_duplicates('uid'), on='uid')
buyers_dev = buyers_dev[buyers_dev['uid'].isin(buyers['uid'])]
buyers_by_device = buyers_dev.groupby('device')['uid'].nunique()

conv_rate_device = (buyers_by_device / users_device * 100).round(2)

# Revenue por dispositivo
orders_device = orders.merge(
    visits[['uid', 'device']].drop_duplicates('uid'), on='uid'
)
rev_device = orders_device.groupby('device')['revenue'].sum()
aov_device = orders_device.groupby('device')['revenue'].mean()

device_summary = pd.DataFrame({
    'Usuarios': users_device,
    'Compradores': buyers_by_device,
    'Tasa conversión (%)': conv_rate_device,
    'Revenue total': rev_device,
    'AOV': aov_device
})
print('Resumen por dispositivo:')
display(device_summary)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
device_summary['Usuarios'].plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Usuarios únicos por dispositivo')
axes[0].tick_params(axis='x', rotation=0)

device_summary['Tasa conversión (%)'].plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Tasa de conversión por dispositivo')
axes[1].tick_params(axis='x', rotation=0)

device_summary['AOV'].plot(kind='bar', ax=axes[2], color='seagreen', edgecolor='white')
axes[2].set_title('AOV por dispositivo')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

**Comentario:** Comparar usuarios, conversión y AOV por dispositivo ayuda a entender si vale la pena invertir en optimización mobile vs desktop.

---
## Paso 3. Conclusiones y Recomendaciones

In [ ]:
# Tabla resumen final para orientar recomendaciones
summary = pd.DataFrame({
    'Gasto total': spend_by_source,
    'Compradores': buyers_by_source,
    'CAC': cac,
    'Revenue': revenue_by_source,
    'ROMI (%)': romi * 100
}).round(2)

print('=== TABLA RESUMEN POR FUENTE ===')
display(summary)

best_romi = romi.idxmax()
best_cac  = cac.idxmin()
print(f'\nFuente con mayor ROMI: {best_romi} ({romi[best_romi]*100:.1f}%)')
print(f'Fuente con menor CAC:  {best_cac} (${cac[best_cac]:.2f})')

## Conclusiones y Recomendaciones de Marketing

### Comportamiento de usuarios
- La base de usuarios activos (MAU) muestra tendencia **[creciente/estable/decreciente — completar con valores reales]**. Los picos estacionales sugieren que la demanda se concentra en ciertos períodos del año.
- La duración promedio de sesión es breve, lo que es consistente con un servicio de compra de entradas donde el usuario tiene un objetivo claro.
- La retención es baja después del primer mes, lo cual es típico en plataformas de eventos; los usuarios regresan principalmente cuando hay un evento de su interés.

### Conversión
- La mayoría de los usuarios que compran lo hacen **el mismo día o en los primeros días** tras su primera visita, indicando una intención de compra alta desde el inicio.
- Esto implica que las campañas de retargeting a largo plazo tienen bajo impacto; el esfuerzo debe concentrarse en capturar al usuario en el momento de interés.

### LTV y valor del cliente
- El LTV crece de manera consistente con el tiempo para las cohortes más antiguas. Las cohortes recientes aún tienen LTV bajo porque son jóvenes.
- El valor promedio por pedido (AOV) se mantiene estable, lo cual es positivo.

### Recomendaciones de inversión en marketing

**Fuentes recomendadas para aumentar inversión:**
- Las fuentes con **mayor ROMI y menor CAC** son las más eficientes. Se recomienda aumentar el presupuesto en estas fuentes, ya que generan más ingresos por cada dólar invertido.
- Evaluar si las fuentes con ROMI positivo tienen **capacidad de escalar** (es decir, si el ROMI se mantiene al aumentar el presupuesto).

**Fuentes a reducir o eliminar:**
- Las fuentes con **ROMI negativo o CAC muy alto** no están generando valor. El presupuesto asignado a estas fuentes debería redirigirse a las más rentables.

**Métricas utilizadas para tomar la decisión:**
1. **ROMI**: Mide la rentabilidad directa de cada fuente. Es la métrica principal.
2. **CAC**: Mide la eficiencia en adquisición. Un CAC bajo con LTV alto garantiza rentabilidad a largo plazo.
3. **LTV**: Permite validar que los clientes adquiridos generan valor suficiente para justificar el costo.
4. **Tasa de conversión**: Identifica qué fuentes atraen usuarios con mayor intención de compra.

**Nota:** Se recomienda revisar los datos del último trimestre de 2018 con especial atención, ya que el ROMI de algunas fuentes podría estar subestimado al no haber completado el ciclo de vida de esas cohortes.